In [ ]:
import os

import matplotlib
import numpy as np
from cil.framework import AcquisitionData, AcquisitionGeometry
from cil.io import TIFFStackReader
from cil.recon import FBP
from cil.utilities.display import show2D, show_geometry
from cil.utilities.jupyter import islicer
from cil.plugins.astra import ProjectionOperator
from gvxrPython3 import gvxr

# CT reconstruction
from gvxrPython3.utils import (
    applyFiltration,
    loadSpectrum,
)
from matplotlib import pyplot as plt
from matplotlib.colors import LogNorm  # Look up table

# Configure matplotlib graph
font = {"family" : "serif",
         "size"   : 15
       }
matplotlib.rc("font", **font)

# Uncomment the line below to use LaTeX fonts
# matplotlib.rc('text', usetex=True)

### Global variables

In [ ]:
OUTPUT_PATH = "../../output_data/beam-hardening/"

if not os.path.exists(OUTPUT_PATH):
    os.makedirs(OUTPUT_PATH)

### Create an OpenGL context

In [ ]:
gvxr.createOpenGLContext()

### Create an iron cylinder as main object source

###  Set up the detector

In [ ]:
# Detector Position and direction
gvxr.setDetectorPosition(20.0, 0.0, 0.0, "cm")
gvxr.setDetectorUpVector(0, 0, -1)

# Detector Pixel definitions
gvxr.setDetectorNumberOfPixels(640, 640)
gvxr.setDetectorPixelSize(0.5, 0.5, "mm")

### Create a source

In [ ]:
# Source position
gvxr.setSourcePosition(
    -20.0,  0.0, 0.0,
    "cm",
)

# Source type
gvxr.useParallelBeam()    # For a parallel source

In [ ]:
def getPolySpectrum(
        tube_voltage_kV: float,
        filters=None,
        tube_angle_in_deg: float=12,
        mAs = None,
        unit = "keV",
) -> dict:

    gvxr.clearFiltration()
    gvxr.setVoltage(tube_voltage_kV, "kV")
    gvxr.setTubeAngle(tube_angle_in_deg)

    if mAs:
        gvxr.setmAs(mAs)
    else:
        gvxr.setmAs(-1)

    if filters:
        applyFiltration(filters)

    energy_bins = gvxr.getEnergyBins(unit)

    photon_count = np.array(gvxr.getPhotonCountsPerCm2At1m(), dtype=np.single)
    # total_photon_count = photon_count.sum()
    photon_count /= photon_count.sum()

    return loadSpectrum(energy_bins, photon_count, unit, False)

In [ ]:
tube_voltage_kV = 100
tube_angle_degrees = 12
max_number_of_energy_bins = 100

# No filter
# ---------
no_filter_hist = getPolySpectrum(
    tube_voltage_kV,
    filters=None,
    tube_angle_in_deg=12,
)
no_filter_bins, no_filter_photons = zip(*no_filter_hist.items(), strict=False)

# Al filters
# ----------
al_half_mm_filter_hist = getPolySpectrum(
    tube_voltage_kV,
    filters=[["Al", 0.5, "mm"]],
    tube_angle_in_deg=12,
)

al_1_mm_filter_hist = getPolySpectrum(
    tube_voltage_kV,
    filters=[["Al", 1.0, "mm"]],
    tube_angle_in_deg=12,
)

al_half_mm_filter_bins, al_half_mm_filter_photons = zip(*al_half_mm_filter_hist.items(), strict=False)
al_1_mm_filter_bins, al_1_mm_filter_photons = zip(*al_1_mm_filter_hist.items(), strict=False)

# Cu filters
# ----------
cu_half_mm_filter_hist = getPolySpectrum(
    tube_voltage_kV,
    filters=[["Cu", 0.5, "mm"]],
    tube_angle_in_deg=12,
)

cu_1_mm_filter_hist = getPolySpectrum(
    tube_voltage_kV,
    filters=[["Cu", 1.0, "mm"]],
    tube_angle_in_deg=12,
)

cu_half_mm_filter_bins, cu_half_mm_filter_photons = zip(*cu_half_mm_filter_hist.items(), strict=False)
cu_1_mm_filter_bins, cu_1_mm_filter_photons = zip(*cu_1_mm_filter_hist.items(), strict=False)

# Sn filters
# ----------
sn_half_mm_filter_hist = getPolySpectrum(
    tube_voltage_kV,
    filters=[["Sn", 0.5, "mm"]],
    tube_angle_in_deg=12,
)

sn_1_mm_filter_hist = getPolySpectrum(
    tube_voltage_kV,
    filters=[["Sn", 1.0, "mm"]],
    tube_angle_in_deg=12,
)

sn_half_mm_filter_bins, sn_half_mm_filter_photons = zip(*sn_half_mm_filter_hist.items(), strict=False)
sn_1_mm_filter_bins, sn_1_mm_filter_photons = zip(*sn_1_mm_filter_hist.items(), strict=False)

In [ ]:
# Plot all the spectra
fig = plt.figure(figsize= (20,10), constrained_layout=True)
fig.supxlabel("Energy (keV)")
fig.supylabel("Probability distribution of photons per keV")
# TODO: Add figure title with the overall filters

plt.subplot(131)
plt.title("Al (Z=13)")
plt.step(no_filter_bins,no_filter_photons,label="No filter")
plt.step(al_half_mm_filter_bins,al_half_mm_filter_photons,label="0.5 mm filter")
plt.step(al_1_mm_filter_bins,al_1_mm_filter_photons,label="1 mm filter")
plt.legend()

plt.subplot(132)
plt.title("Cu (Z=29)")
plt.step(no_filter_bins,no_filter_photons,label="No filter")
plt.step(cu_half_mm_filter_bins, cu_half_mm_filter_photons,label="0.5 mm filter")
plt.step(cu_1_mm_filter_bins, cu_1_mm_filter_photons,label="1 mm filter")
plt.legend()

plt.subplot(133)
plt.title("Sn (Z=50)")
plt.step(no_filter_bins,no_filter_photons,label="No filter")
plt.step(sn_half_mm_filter_bins,sn_half_mm_filter_photons,label="0.5 mm filter")
plt.step(sn_1_mm_filter_bins,sn_1_mm_filter_photons,label="1 mm filter")
plt.legend()

plt.show()

In [ ]:
gvxr.removePolygonMeshesFromSceneGraph()

cylinder_height = 20
cylinder_radius = 10

gvxr.makeCylinder(
    "Cylinder",
    24, # Number of sectors
    cylinder_height, cylinder_radius,
    "cm",
)

gvxr.addPolygonMeshAsOuterSurface("Cylinder")
gvxr.setElement("Cylinder", "Fe")

gvxr.rotateNode("Cylinder", 0, 0, 90)

In [ ]:
x_ray_image = np.array(gvxr.computeXRayImage(), dtype=np.single) / gvxr.getTotalEnergyWithDetectorResponse()

plt.imshow(x_ray_image, norm=LogNorm(), cmap="gray")
plt.title("X-ray Image")
plt.colorbar(orientation="horizontal")

plt.show()

### Compute CT Acquisition Arguments
| Function Argument | Description |
|-------------------|-------------|
| `projection_output_path` | The path where the X-ray projections will be saved. If path is empty, data will be stored in main memory, but not saved on the disk. If path is provided, the data will be saved on the disk, and the main memory released. |
| `screenshot_output_path` | The path where the screenshots will be saved. If kept empty, not screenshot will be saved. |
| `num_of_projections` | The total number of projections to simulate. |
| `first_angle` | The rotation angle corresponding to the first projection. |
| `include_last_angle` | A boolean flag to include or exclude the last angle. It is used to calculate the angular step between successive projections. |
| `last_angle` | The number of white images used to perform the flat-field correction. If zero, then no correction will be performed. |
| `num_of_white_images_in_flat_field` | The location of the rotation centre. |
| `position_of_centre_of_rotation` | The corresponding unit of length. |
| `unit_of_length` | The rotation axis |
| `axis_of_rotation` | The upvector |
| `integrate_energy` | If true the energy fluence is returned, otherwise the number of photons is returned (default value: true) |

In [ ]:
getPolySpectrum(
    tube_voltage_kV,
    filters=None,
    tube_angle_in_deg=12,
)

projection_output_path = os.path.join(OUTPUT_PATH, "cylinder-recons")
screenshot_output_path = ""
num_of_projections = 360
first_angle = 0
include_last_angle = False
last_angle = 360
num_of_white_images_in_flat_field = 0
position_of_centre_of_rotation = (0, 0, 0)
unit_of_length = "cm"
axis_of_rotation = (0, 0, -1)
integrate_energy = True

gvxr.computeCTAcquisition(
    projection_output_path,
    screenshot_output_path,
    num_of_projections,
    first_angle,
    include_last_angle,
    last_angle,
    num_of_white_images_in_flat_field,
    *position_of_centre_of_rotation, "mm",
    *axis_of_rotation,
    integrate_energy,
)

# TODO: Show a widget for the user to select a filter with thickness to be able to compute a CT scan

In [ ]:
# Create the TIFF reader by passing the directory containing the files
reader = TIFFStackReader(
    file_name=os.path.join(OUTPUT_PATH, "cylinder-recons"),
    dtype=np.float32,
)

# Read in file, and return a numpy array containing the data
data_original = reader.read()

# The data is stored as a stack of detector images, we use the CILlabels for the axes
axis_labels = [
    "angle",
    "vertical",
    "horizontal"
]

# Normalisation
# Not strictly needed as the data was already corrected
data_normalised = data_original / data_original.max()

# Prevent log of a negative value
data_normalised[data_normalised < 1e-9] = 1e-9

# Linearisation
data_absorption = -np.log(data_normalised)

In [ ]:
geometry = AcquisitionGeometry.create_Parallel3D(
    ray_direction=[1,0,0],
    detector_position=gvxr.getDetectorPosition("cm"),
    detector_direction_x=gvxr.getDetectorRightVector(),
    detector_direction_y=gvxr.getDetectorUpVector(),
    rotation_axis_position=gvxr.getCentreOfRotationPositionCT("cm"),
    rotation_axis_direction=gvxr.getRotationAxisCT(),
)

# Set the angles, remembering to specify the units
geometry.set_angles(
    np.array(gvxr.getAngleSetCT()),
    angle_unit="degree",
)

# Set the detector shape and size
geometry.set_panel(
    gvxr.getDetectorNumberOfPixels(),
    gvxr.getDetectorPixelSpacing("cm"),
)

# Set the order of the data
geometry.set_labels(axis_labels)

Working on the CPU, we cannot reconstruct the whole 3D volume for a cone beam geometry. Instead we reconstruct a single slice:

In [ ]:
# Prepare the data for the reconstruction
acquisition_data = AcquisitionData(
    data_absorption,
    deep_copy=False,
    geometry=geometry,
)

ig = acquisition_data.geometry.get_ImageGeometry()

# get slice
data_slice = acquisition_data.get_slice(vertical='centre')

#  set up the 2D fandbeam geometry
ig_2d = data_slice.geometry.get_ImageGeometry()

```{warning}
An NVIDIA GPU is required for the reconstruction using CIL
```

In [ ]:
A = ProjectionOperator(ig_2d, data_slice.geometry, device='cpu')

# Perform the FDK reconstruction
fbp =  FBP(acquisition_data, ig_2d)
recon = fbp.run()

In [ ]:
show2D(recon)

In [ ]:
from cil.utilities.display import show1D

show1D(recon,title="line profile of slices")